In [ ]:
import warnings

warnings.filterwarnings("ignore")
from pathlib import Path  # noqa: E402

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import seaborn as sns  # noqa: E402
from scipy import stats  # noqa: E402, F401
from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.preprocessing import LabelEncoder  # noqa: E402

In [ ]:
from src.data.data_loader import load_dataset

In [ ]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUT_DIR = PROJECT_ROOT / "data" / "eda" / "eda_graphics"

  > Что мы знаем о raw данных из описания датасета?
 
 | 1 | 2 | 3 | 4|
  |---|---| ---|---|
  | PS1 | 100Hz | shape=(2205, 6000) | Pressure supply |
  | PS2 |  100Hz  | shape=(2205, 6000) | Pressure return| 
  | PS3  |   100Hz | shape=(2205, 6000)|  Pressure pump |
  | PS4   |  100Hz | shape=(2205, 6000) | Pressure servo |
  | PS5  |   100Hz  | shape=(2205, 6000)  | Pressure cylinder |
  | PS6   |  100Hz  | shape=(2205, 6000)  | Pressure cooler exit |
  | EPS1  |  100Hz  | shape=(2205, 6000)  | Motor power | 
  | FS1   |   10Hz  | shape=(2205, 600)  | Flow pump |
  | FS2   |   10Hz  |shape=(2205, 600)  | Flow return |
  | TS1   |    1Hz | shape=(2205, 60)  | Temp supply |
 | TS2   |    1Hz  | shape=(2205, 60)  | Temp return |
  | TS3    |   1Hz  | shape=(2205, 60)  | Temp pump |
  | TS4   |    1Hz  | shape=(2205, 60)  | Temp cooler exit |
  | VS1   |   1Hz | shape=(2205, 60)  | Vibration |
  | CE    |    1Hz | shape=(2205, 60)  | Cooling efficiency |
  | CP    |    1Hz | shape=(2205, 60)  | Cooling power|
  | SE   |     1Hz | shape=(2205, 60)  | System efficiency |


 > Total cycles: 2205

In [ ]:
SENSOR_DESCRIPTIONS = {
    "PS1": "Pressure supply",
    "PS2": "Pressure return",
    "PS3": "Pressure pump",
    "PS4": "Pressure servo",
    "PS5": "Pressure cylinder",
    "PS6": "Pressure cooler exit",
    "EPS1": "Motor power",
    "FS1": "Flow pump",
    "FS2": "Flow return",
    "TS1": "Temperature supply",
    "TS2": "Temperature return",
    "TS3": "Temperature pump",
    "TS4": "Temperature cooler exit",
    "VS1": "Vibration",
    "CE": "Cooling efficiency",
    "CP": "Cooling power",
    "SE": "System efficiency",
}

TARGET_DESCRIPTIONS = {
    "cooler": "Cooler condition",
    "valve": "Valve condition",
    "pump": "Pump leakage",
    "accumulator": "Accumulator pressure",
}

In [ ]:
sensors, profile = load_dataset(DATA_DIR)

In [ ]:
features = pd.read_csv("../data/processed/X.csv")
features.describe()

In [ ]:
df = features.merge(
    profile,
    left_index=True,
    right_index=True,
)

print(f"DataFrame: {df.shape}")
df.head()

In [ ]:
# Проверка объединённого датасета
print("DataFrame (features + profile):")
print(f"  Shape: {df.shape}")
print(f"  NaN: {df.isna().sum().sum()}")
print(f"  Inf: {np.isinf(df.select_dtypes(include=[np.number])).sum().sum()}")
print()

# Дубликаты
n_duplicates = df.duplicated().sum()
print(f"  Duplicates: {n_duplicates}")

In [ ]:
df.describe().T

### 1. Label distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle("Condition label distributions", fontsize=13, fontweight="bold")

plot_info = [
    ("cooler", axes[0, 0], "#640505", {3: "Near failure", 20: "Reduced", 100: "Full"}),
    (
        "valve",
        axes[0, 1],
        "#BC8232",
        {73: "Near failure", 80: "Severe lag", 90: "Small lag", 100: "Optimal"},
    ),
    ("pump", axes[1, 0], "#4E91D4", {0: "No leakage", 1: "Weak", 2: "Severe"}),
    (
        "accumulator",
        axes[1, 1],
        "#1D9E75",
        {90: "Near failure", 100: "Severe", 115: "Slight", 130: "Optimal"},
    ),
]

for col, ax, color, lmap in plot_info:
    vc = df[col].value_counts().sort_index()
    lbls = [lmap.get(v, str(v)) for v in vc.index]
    bars = ax.bar(lbls, vc.values, color=color, alpha=0.8, edgecolor="white")
    ax.set_title(col.capitalize(), fontweight="bold")
    ax.set_ylabel("Cycles")
    ax.tick_params(axis="x", rotation=20)
    for b, v in zip(bars, vc.values):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 5, str(v), ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig01_label_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
FEATURES = [
    "PS1_mean",
    "PS2_mean",
    "PS4_mean",
    "PS5_mean",
    "TS1_mean",
    "TS4_mean",
    "FS1_mean",
    "FS2_mean",
    "SE_mean",
    "CE_mean",
    "EPS1_mean",
    "PS1_std",
]

LABELS = {
    "cooler": {3: "Near failure", 20: "Reduced", 100: "Full"},
    "valve": {73: "Near failure", 80: "Severe lag", 90: "Small lag", 100: "Optimal"},
    "pump": {0: "No leakage", 1: "Weak", 2: "Severe"},
    "accumulator": {90: "Near failure", 100: "Severe", 115: "Slight", 130: "Optimal"},
}

COLORS = {
    "cooler": ["#E24B4A", "#BA7517", "#1D9E75"],
    "valve": ["#E24B4A", "#BA7517", "#378ADD", "#1D9E75"],
    "pump": ["#1D9E75", "#BA7517", "#E24B4A"],
    "accumulator": ["#E24B4A", "#BA7517", "#378ADD", "#1D9E75"],
}

SENSOR_UNITS = {
    "PS1": "bar",
    "PS2": "bar",
    "PS3": "bar",
    "PS4": "bar",
    "PS5": "bar",
    "PS6": "bar",
    "EPS1": "W",
    "FS1": "l/min",
    "FS2": "l/min",
    "TS1": "°C",
    "TS2": "°C",
    "TS3": "°C",
    "TS4": "°C",
    "VS1": "mm/s",
    "CE": "uS/cm",
    "CP": "bar",
    "SE": "%",
}

In [ ]:
def plot_dist(df, target, features, out_dir):  # noqa: ANN001, ANN201
    """Boxplots: распределения признаков по классам таргета."""
    y = df[target].map(LABELS[target])
    order = list(LABELS[target].values())

    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    fig.suptitle(f"Feature distributions by {target}", fontsize=13, fontweight="bold")

    for feat, ax in zip(features, axes.flat):
        sns.boxplot(
            x=y,
            y=df[feat],
            order=order,
            palette=COLORS[target],
            ax=ax,
            width=0.6,
        )
        ax.set_title(feat, fontsize=9)
        ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=15, labelsize=7)
        ax.grid(alpha=0.3, axis="y")

    for ax in axes.flat[len(features) :]:
        ax.remove()

    plt.tight_layout()
    path = out_dir / f"fig03_{target}.png"
    fig.savefig(path, dpi=120, bbox_inches="tight")
    # для удобства только один график вывести
    if target == "cooler":
        plt.show()
    plt.close(fig)
    return path


# Запуск
for target in ["cooler", "valve", "pump", "accumulator"]:
    plot_dist(df, target, FEATURES, OUT_DIR)

In [ ]:
PLOT_CFG = [
    ("cooler", "PS1"),
    ("cooler", "TS1"),
    ("cooler", "TS4"),
    ("valve", "PS2"),
    ("valve", "FS2"),
    ("valve", "PS4"),
    ("pump", "PS2"),
    ("pump", "SE"),
    ("pump", "EPS1"),
    ("accumulator", "PS4"),
    ("accumulator", "PS5"),
    ("accumulator", "VS1"),
]

fig, axes = plt.subplots(4, 3, figsize=(16, 14))
fig.suptitle("Mean sensor signal by component condition", fontsize=13, fontweight="bold")

for (target, sensor), ax in zip(PLOT_CFG, axes.flat):
    lbls = LABELS[target]
    colors = COLORS[target]
    order = sorted(lbls.keys())

    for i, cv in enumerate(order):
        mask = (profile[target] == cv).values
        sig = sensors[sensor][mask].mean(axis=0)
        t = np.linspace(0, 60, len(sig))
        ax.plot(t, sig, color=colors[i], label=lbls[cv], linewidth=1.8, alpha=0.9)

    sensor_desc = SENSOR_DESCRIPTIONS.get(sensor, sensor)
    target_desc = TARGET_DESCRIPTIONS.get(target, target)
    ax.set_title(f"{sensor_desc} ({sensor}) by {target_desc}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel(SENSOR_UNITS.get(sensor, ""))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "fig02_sensor_signals.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 1. Все mean-признаки
mean_cols = [c for c in df.columns if c.endswith("_mean")]
corr = df[mean_cols].corr()

# 2. Heatmap — все со всеми
fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool), k=0)  # ← верхний треугольник

sns.heatmap(
    corr,
    mask=mask,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 7},
    linewidths=0.3,
    square=True,
    ax=ax,
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Sensor mean correlations", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "fig04_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Топ-15 по |corr| без NaN и без пар с собой
corr_abs_pairs = corr.where(~mask).stack().dropna().sort_values(key=abs, ascending=False)

corr_pairs = corr.where(~mask).stack().dropna().sort_values(ascending=False)

print("  Топ-7 положительно коррелирующих пар")
print(corr_abs_pairs.head(7).to_string(), "\n")

print("  Топ-7 отрицательно коррелирующих пар")
print(corr_pairs.tail(7).to_string(), "\n")


print("  Топ-7 анти-коррелирующих пар")
print(corr_abs_pairs.tail(7).to_string(), "\n")

In [ ]:
def plot_feature_importance(df, target, output_dir, top_n=20) -> None:  # noqa: ANN001
    """Bar chart: feature importance для одного таргета."""
    # Признаки без таргетов
    target_cols = ["cooler", "valve", "pump", "accumulator", "stable", "cycle_id"]
    X_feat = df.drop(columns=target_cols, errors="ignore")  # noqa: N806

    # Таргет
    y_enc = LabelEncoder().fit_transform(df[target])

    # RF
    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_feat, y_enc)

    # Топ-N
    imp = pd.Series(rf.feature_importances_, index=X_feat.columns).nlargest(top_n).sort_values()

    # График
    fig, ax = plt.subplots(figsize=(10, 7))
    imp.plot.barh(ax=ax, color="#185FA5", alpha=0.85)
    ax.set_title(f"Top-{top_n} features — RF importance for {target}", fontweight="bold")
    ax.set_xlabel("Importance")
    ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    path = output_dir / f"fig05_feature_importance_{target}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    if target == "cooler":
        plt.show()
    plt.close(fig)


# Запуск для всех таргетов
for target in ["cooler", "valve", "pump", "accumulator"]:
    plot_feature_importance(df, target, OUT_DIR)

In [ ]:
from pathlib import Path

for path in sorted(OUT_DIR.glob("*.png")):
    print(path.name)

In [ ]:
import hashlib
from pathlib import Path

import mlflow

# MLflow: EDA artifacts & dataset tracking
MLFLOW_TRACKING_URI = "http://localhost:5001"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("hydraulic-eda")


# Helper: SHA-256


def dataframe_sha256(df):  # noqa: ANN001, ANN201
    """Посчитать SHA-256 для содержимого DataFrame."""
    data = pd.util.hash_pandas_object(
        df,
        index=True,
    ).values.tobytes()

    return hashlib.sha256(data).hexdigest()


# Dataset information

dataset_digest = dataframe_sha256(df)

dataset_metadata = {
    "source": "UCI Condition Monitoring of Hydraulic Systems",
    "dataset_version": "UCI Dataset 447",
    "n_samples": int(df.shape[0]),
    "n_features_total": int(df.shape[1]),
    "feature_columns": list(features.columns),
    "target_columns": list(profile.columns),
    "feature_pipeline": [
        "data_loader.py",
        "features.py",
    ],
    "dataset_digest": dataset_digest,
    "lineage": ("UCI raw sensor files → data_loader.py → features.py → features + profile → EDA"),
}


# MLflow Run

with mlflow.start_run(run_name="eda"):
    # Dataset tracking

    mlflow.set_tags(
        {
            "dataset.source": dataset_metadata["source"],
            "dataset.version": dataset_metadata["dataset_version"],
            "dataset.digest": dataset_digest,
            "dataset.lineage": dataset_metadata["lineage"],
        }
    )

    mlflow.log_params(
        {
            "n_samples": dataset_metadata["n_samples"],
            "n_features_total": dataset_metadata["n_features_total"],
            "n_features": len(features.columns),
            "n_targets": len(profile.columns),
        }
    )

    # Dataset metadata

    mlflow.log_dict(
        dataset_metadata,
        "dataset/dataset_metadata.json",
    )

    # Existing EDA plots

    plot_paths = sorted(OUT_DIR.glob("*.png"))

    if not plot_paths:
        raise RuntimeError(f"В {OUT_DIR} не найдено PNG-файлов")

    for path in plot_paths:
        mlflow.log_artifact(
            str(path),
            artifact_path="eda",
        )

    # Target statistics

    target_statistics = profile.describe().T

    target_statistics_path = OUT_DIR / "target_statistics.csv"

    target_statistics.to_csv(target_statistics_path)

    mlflow.log_artifact(
        str(target_statistics_path),
        artifact_path="eda",
    )

In [ ]:
# Summary
print("MLflow EDA run completed")

print("\nExperiment: hydraulic-eda")
print("Run:        eda")

print("\nDataset:")
print(f"  samples:        {df.shape[0]}")
print(f"  total columns:  {df.shape[1]}")
print(f"  features:       {features.shape[1]}")
print(f"  targets:        {profile.shape[1]}")

print("\nDataset digest:")
print(f"  {dataset_digest}")

print(f"\nEDA artifacts: {len(plot_paths)} plots")

for path in plot_paths:
    print(f"  {path.name}")

print(f"  {target_statistics_path.name}")
print("  dataset_metadata.json")